# Rurality, density and job reallocation (v1)

Do more rural / less dense regions have less job reallocation, and why? Uses the LBD job flows (`clean/itl2_2021/job_flows_itl2.csv`) with the settlement-structure measures built in `build/rurality.ipynb` and `build/density.ipynb`, all on ITL2 **2021** (41 regions).

**Why it matters:** thick local labour and product markets make it easier for workers and jobs to move between firms (matching and agglomeration: e.g. Duranton & Puga 2004; Moretti 2011), so denser places should show more reallocation. Rural places may also have a different industry mix, more small or single-site firms, and fewer outside options.

**Measures of rurality (x-axis) and why the main one is population-weighted density:**

| measure | what | issue |
|---|---|---|
| **Population-weighted density** (PWD, 2021) | density of the small area the average resident lives in (persons/km²) | **main measure**: same method across the UK; continuous; log scale (spans ~1,500 to ~15,000) |
| Area density (2021) | residents ÷ land area | driven by empty land, but a useful check |
| Share of population in rural areas | Census-based official classifications | **definitions differ by nation**: England & Wales < 10k, Scotland < 3k, NI < 5k. Reported for all regions and for England & Wales only |

**Method choices:**
- **Window:** pooled **2011–2024** (14 years): the longest window in which all 41 regions, including Northern Ireland, have reliable data. Density barely changes, so this is a cross-sectional comparison and a long window smooths out the cycle and the 2010/2023 spikes. The window choice makes little difference (see the robustness table). Reallocation is pooled from counts: (Σ JC + Σ JD) / Σ average employment.
- **Robustness:** with and without London; Spearman ranks; England & Wales only for the rural share; **partial correlation controlling for productivity** (rural regions are also less productive, which the productivity analysis showed is linked to churn); full 2003–2024 window; dropping spike and COVID years; each 5-year window.
- **Mechanism:** which flows differ: firm **entry and exit**, incumbents **opening and closing sites**, or incumbents' sites **expanding and contracting**.
- Descriptive, not causal: about 40 regions, and density is fixed over time, so there is no within-region variation to exploit.

Charts are saved to `analysis/rurality/charts/` (JSON + PNG); shared chart setup is in `analysis/chart_setup.py`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy import stats

BASE  = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")
DIR21 = BASE / "clean" / "itl2_2021"
FIG   = BASE / "analysis" / "rurality" / "charts"
FIG.mkdir(parents=True, exist_ok=True)
sys.path.append(str(BASE / "analysis"))
from chart_setup import styles, alt, ECO, TXT, GREY, place_labels

SOURCE = ("Sources: ONS Longitudinal Business Database (Secure Access, SN 9501); ONS, NRS and NISRA population estimates; "
          "authors' calculations.")
MAIN = (2011, 2024)
FLOWS = ["jc_entry", "jc_incumbent_ext", "jc_incumbent_int", "jd_incumbent_int", "jd_incumbent_ext", "jd_exit"]
COMPONENTS = {"Entry and exit": ["jc_entry", "jd_exit"],
              "Site openings and closures": ["jc_incumbent_ext", "jd_incumbent_ext"],
              "Expansion and contraction": ["jc_incumbent_int", "jd_incumbent_int"]}

reg = pd.read_csv(DIR21 / "job_flows_itl2.csv")
jf = reg[reg.analysis_window & ~((reg.itl2_code == "TLN0") & (reg.year < 2011))].copy()   # NI's 2003-10 series is erratic
LONDON = lambda d: d.itl2_code.str.startswith("TLI")

def pooled(years, drop=()):
    """Pooled annual rates (% of average employment) from summed counts, per region."""
    d = jf[jf.year.between(*years) & ~jf.year.isin(drop)]
    g = d.groupby("itl2_code")[FLOWS + ["total_jc", "total_jd", "emp_avg"]].sum()
    out = pd.DataFrame({"realloc": 100 * (g.total_jc + g.total_jd) / g.emp_avg, "emp": g.emp_avg / d.groupby("itl2_code").year.nunique()})
    for k, cols in COMPONENTS.items():
        out[k] = 100 * g[cols].sum(axis=1) / g.emp_avg
    return out

# settlement measures (2021) and productivity (control)
ru  = pd.read_csv(DIR21 / "rurality_itl2.csv").set_index("itl2_code")
pwd = pd.read_csv(DIR21 / "pop_weighted_density_itl2.csv").query("year == 2021").set_index("itl2_code")
pr  = pd.read_csv(DIR21 / "productivity_itl2.csv").query("hours_worked_weekly.notna() and year >= 2011")
pr  = pr.assign(gva=pr.gva_per_hour_gbp * pr.hours_worked_weekly)
uk_ph = pr.groupby("year").gva.sum() / pr.groupby("year").hours_worked_weekly.sum()
pr  = pr.assign(at_uk=pr.hours_worked_weekly * pr.year.map(uk_ph))
prod_rel = (100 * pr.groupby("itl2_code").gva.sum() / pr.groupby("itl2_code").at_uk.sum()).rename("prod_rel")   # 2011-23, UK = 100

def frame(years, drop=()):
    d = (pooled(years, drop)
         .join(ru[["itl2_name", "share_rural", "rural_definition"]])
         .join(pwd[["pop_weighted_density", "area_density"]]).join(prod_rel).reset_index())
    d["log_pwd"], d["log_area_density"] = np.log(d.pop_weighted_density), np.log(d.area_density)
    d["group"] = np.where(LONDON(d), "London", "Rest of UK")
    return d

df = frame(MAIN)
assert len(df) == 41 and df.notna().all().all()
print(f"{len(df)} regions; pooled {MAIN[0]}–{MAIN[1]}; PWD range {df.pop_weighted_density.min():,.0f}–{df.pop_weighted_density.max():,.0f} per km²")

41 regions; pooled 2011–2024; PWD range 1,543–14,810 per km²


## 1. How strongly is rurality linked to reallocation?

Pearson r, Spearman ρ and the **partial correlation controlling for log productivity** (GVA per hour relative to the UK, 2011–23) for each measure and sample. A positive r for density, or a negative r for the rural share, means denser or more urban regions have more reallocation.

In [2]:
def partial(x, y, z):
    """Correlation of x and y after removing the linear effect of z from both."""
    res = lambda v: v - np.polyval(np.polyfit(z, v, 1), z)
    return np.corrcoef(res(x), res(y))[0, 1]

MEASURES = {"log_pwd": "Pop.-weighted density (log)", "log_area_density": "Area density (log)", "share_rural": "Rural share (%)"}
SAMPLES = {"All regions": lambda d: d, "Excl. London": lambda d: d[~LONDON(d)],
           "England & Wales, excl. London": lambda d: d[~LONDON(d) & d.rural_definition.str.startswith("E&W")]}
rows = []
for sname, f in SAMPLES.items():
    s = f(df)
    for m, mname in MEASURES.items():
        rows.append({"sample": sname, "measure": mname, "n": len(s), "Pearson r": np.corrcoef(s.realloc, s[m])[0, 1],
                     "Spearman rho": stats.spearmanr(s.realloc, s[m])[0],
                     "partial r | productivity": partial(s.realloc.values, s[m].values, np.log(s.prod_rel.values))})
corr = pd.DataFrame(rows)
print("Reallocation (pooled 2011–24) vs settlement measures"); print(corr.round(2).to_string(index=False))

# robustness: other windows and dropping spike/COVID years (log PWD, excl. London)
rob = []
for lab, yrs, drop in [("2011–24 (main)", MAIN, ()), ("2011–24 excl. 2020, 2021, 2023", MAIN, (2020, 2021, 2023)),
                       ("2003–24 (NI from 2011)", (2003, 2024), ()), ("2004–08", (2004, 2008), ()), ("2009–13", (2009, 2013), ()),
                       ("2014–18", (2014, 2018), ()), ("2019–23", (2019, 2023), ())]:
    d = frame(yrs, drop); s = d[~LONDON(d)]
    rob.append({"window": lab, "r all": np.corrcoef(d.realloc, d.log_pwd)[0, 1], "r excl. London": np.corrcoef(s.realloc, s.log_pwd)[0, 1],
                "rho excl. London": stats.spearmanr(s.realloc, s.log_pwd)[0]})
print("\nRobustness: reallocation vs log PWD"); print(pd.DataFrame(rob).round(2).to_string(index=False))

# mechanism: which flows are linked to density (excl. London)
s = df[~LONDON(df)]
print("\nCorrelation of each pair of flows with log PWD, excl. London (pooled 2011–24, % of employment):")
for k in COMPONENTS:
    print(f"  {k:<28} mean {s[k].mean():5.2f}  r = {np.corrcoef(s[k], s.log_pwd)[0, 1]:+.2f}")

Reallocation (pooled 2011–24) vs settlement measures
                       sample                     measure  n  Pearson r  Spearman rho  partial r | productivity
                  All regions Pop.-weighted density (log) 41       0.83          0.70                      0.64
                  All regions          Area density (log) 41       0.87          0.85                      0.82
                  All regions             Rural share (%) 41      -0.71         -0.75                     -0.56
                 Excl. London Pop.-weighted density (log) 36       0.60          0.56                      0.55
                 Excl. London          Area density (log) 36       0.79          0.78                      0.81
                 Excl. London             Rural share (%) 36      -0.64         -0.63                     -0.61
England & Wales, excl. London Pop.-weighted density (log) 30       0.65          0.61                      0.58
England & Wales, excl. London          Area density

## 2. Chart: reallocation against population-weighted density

Log x-axis (density spans an order of magnitude); straight-line fit on the log scale; London in red; labels sparse (tooltips show every region).

In [3]:
# Density scatters, Northern Ireland excluded throughout (its job flows start in 2011), so every period covers the same 40 regions.
FULL = (2003, 2024)
def frame_noni(years):
    d = frame(years); return d[d.itl2_code != "TLN0"].reset_index(drop=True)

def uk_rate(years):
    j = jf[jf.year.between(*years) & (jf.itl2_code != "TLN0")]
    return 100 * j[["total_jc", "total_jd"]].sum().sum() / j.emp_avg.sum()

def fit(dd, XS, YS, colour):
    b = np.polyfit(dd.log_pwd, dd.realloc, 1); fx = np.exp(np.linspace(dd.log_pwd.min(), dd.log_pwd.max(), 50))
    return (alt.Chart(pd.DataFrame({"x": fx, "y": np.polyval(b, np.log(fx))})).mark_line(color=colour, strokeWidth=1.6, opacity=0.65)
              .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS)))

def density_panel(d, years, XS, YS, W, H, first=True, ytitle="Job reallocation rate"):
    uk = uk_rate(years)
    labs = place_labels(d, "pop_weighted_density", "realloc", XS, YS, W, H, xlog=True, max_labels=7, max_len=22)
    ax_y = alt.Axis(title=ytitle if first else None, labels=first, ticks=first, labelFlush=True, labelExpr="datum.label + '%'")
    return alt.layer(
        alt.Chart(pd.DataFrame({"v": [uk]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YS, axis=ax_y)),
        alt.Chart(pd.DataFrame({"x": [XS.domain[1]], "y": [uk]})).mark_text(align="right", baseline="bottom", dx=-3, dy=-3, fontSize=10, color=TXT)
           .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS), text=alt.value(f"UK {uk:.1f}%")),
        fit(d, XS, YS, ECO["red"]), fit(d[~LONDON(d)], XS, YS, ECO["blue-dark"]),
        alt.Chart(d).mark_circle(size=70, opacity=0.85, stroke="white", strokeWidth=0.6).encode(
            x=alt.X("pop_weighted_density:Q", scale=XS, title="Population-weighted density, 2021 (residents per km², log scale)",
                    axis=alt.Axis(values=[1000, 2000, 5000, 10000, 20000], format=",")),
            y=alt.Y("realloc:Q", scale=YS, axis=ax_y),
            color=alt.Color("group:N", scale=alt.Scale(domain=["London", "Rest of UK"], range=[ECO["red"], ECO["blue-light"]]),
                            legend=alt.Legend(title=None, orient="bottom") if first else None),
            tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("pop_weighted_density:Q", format=",.0f", title="Pop.-weighted density"),
                     alt.Tooltip("share_rural:Q", format=".0f", title="Rural share (%)"), alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
        *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=9, color=TXT)
            .encode(x=alt.X("pop_weighted_density:Q", scale=XS), y=alt.Y("realloc:Q", scale=YS), text="short:N") for i, r in labs.iterrows()],
    )

def rs(d):
    o = d[~LONDON(d)]
    return np.corrcoef(d.realloc, d.log_pwd)[0, 1], np.corrcoef(o.realloc, o.log_pwd)[0, 1]

XS = alt.Scale(type="log", domain=[1000, 20000], nice=False)
FITS = "Red line = fit across all regions; navy = excluding London."
NI_NOTE = "Northern Ireland excluded (job flows only from 2011)."

# (a) full period, 2003-24
W, H = 560, 400
dfull = frame_noni(FULL); r_all, r_ex = rs(dfull)
YS = alt.Scale(domain=[float(np.floor(dfull.realloc.min() - 1)), float(np.ceil(dfull.realloc.max() + 1))], zero=False, nice=False)
scat = density_panel(dfull, FULL, XS, YS, W, H).properties(title=alt.Title("Denser regions have more job reallocation, in and outside London",
    subtitle=[f"ITL2 regions, reallocation pooled 2003–24. Correlation {r_all:.2f} across all regions; {r_ex:.2f} excluding London.",
              f"Population-weighted density: density of the small area the average resident lives in. {FITS} {NI_NOTE}", SOURCE]))
styles.save(scat, str(FIG), "reallocation_density", width=W, height=H)
print(f"2003–24 (40 regions): r = {r_all:.2f}, excl. London {r_ex:.2f}")

# (b) two six-year periods: 2004-09 and 2018-23
PERIODS_D = {"2004–09": (2004, 2009), "2018–23": (2018, 2023)}
dp = {lab: frame_noni(yrs) for lab, yrs in PERIODS_D.items()}
allp = pd.concat(dp.values())
YS2 = alt.Scale(domain=[float(np.floor(allp.realloc.min() - 1)), float(np.ceil(allp.realloc.max() + 1))], zero=False, nice=False)
panels_d = []
for i, (lab, yrs) in enumerate(PERIODS_D.items()):
    ra, re_ = rs(dp[lab]); print(f"{lab}: r = {ra:.2f}, excl. London {re_:.2f}")
    panels_d.append(density_panel(dp[lab], yrs, XS, YS2, 360, 360, first=i == 0).properties(width=360, height=360,
        title=alt.Title(lab, subtitle=f"r = {ra:.2f} (excl. London {re_:.2f})", anchor="start", fontSize=13, subtitleFontSize=11)))
scat2 = alt.hconcat(*panels_d, spacing=14).resolve_scale(color="shared").properties(title=alt.Title(
    "The density gap in job churn has persisted as churn fell everywhere",
    subtitle=["ITL2 regions, reallocation pooled over each six-year period, against population-weighted density (2021). Same axes in both panels.",
              f"{FITS} {NI_NOTE}", SOURCE]))
styles.save(scat2, str(FIG), "reallocation_density_periods", width=360, height=360)
scat.properties(width=W, height=H)

2003–24 (40 regions): r = 0.85, excl. London 0.65
2004–09: r = 0.77, excl. London 0.58
2018–23: r = 0.78, excl. London 0.50


alt.LayerChart(...)

In [4]:
# Single scatter: reallocation and population-weighted density both pooled over 2018-24 (all 41 regions; NI has data for these years)
YRS = (2018, 2024)
pw = pd.read_csv(DIR21 / "pop_weighted_density_itl2.csv").query("year >= @YRS[0] and year <= @YRS[1]")
pwd_pool = (pw.assign(w=pw.pop_weighted_density * pw.population).groupby("itl2_code")[["w", "population"]].sum()
              .pipe(lambda g: g.w / g.population).rename("pwd_pool"))          # population-weighted across years
d = pooled(YRS).join(ru[["itl2_name", "share_rural"]]).join(pwd_pool).reset_index()
d["log_pwd"] = np.log(d.pwd_pool); d["group"] = np.where(LONDON(d), "London", "Rest of UK")
assert len(d) == 41 and d.notna().all().all()
r_all = np.corrcoef(d.realloc, d.log_pwd)[0, 1]; o = d[~LONDON(d)]; r_ex = np.corrcoef(o.realloc, o.log_pwd)[0, 1]
uk = 100 * jf[jf.year.between(*YRS)][["total_jc", "total_jd"]].sum().sum() / jf[jf.year.between(*YRS)].emp_avg.sum()
print(f"2018–24, 41 regions: r = {r_all:.2f}, excl. London {r_ex:.2f}; UK {uk:.1f}%")

W, H = 560, 400
XS = alt.Scale(type="log", domain=[1000, 20000], nice=False)
YS = alt.Scale(domain=[float(np.floor(d.realloc.min() - 1)), float(np.ceil(d.realloc.max() + 1))], zero=False, nice=False)
b = np.polyfit(d.log_pwd, d.realloc, 1); fx = np.exp(np.linspace(d.log_pwd.min(), d.log_pwd.max(), 50))
labs = place_labels(d.rename(columns={"pwd_pool": "pop_weighted_density"}), "pop_weighted_density", "realloc", XS, YS, W, H, xlog=True, max_labels=8, max_len=22)
# extra label: densest non-London region below the UK average
extra = d[~LONDON(d) & (d.realloc < uk)].nlargest(1, "pwd_pool").assign(short=lambda x: x.itl2_name)
print("Extra label:", extra.itl2_name.iloc[0], round(extra.pwd_pool.iloc[0]), round(extra.realloc.iloc[0], 1))
labs = labs[~labs.itl2_code.isin(extra.itl2_code)] if len(labs) else labs
ch = alt.layer(
    alt.Chart(pd.DataFrame({"v": [uk]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YS)),
    alt.Chart(pd.DataFrame({"x": [XS.domain[1]], "y": [uk]})).mark_text(align="right", baseline="bottom", dx=-3, dy=-3, fontSize=10, color=TXT)
       .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS), text=alt.value(f"UK {uk:.1f}%")),
    alt.Chart(pd.DataFrame({"x": fx, "y": np.polyval(b, np.log(fx))})).mark_line(color=GREY, strokeWidth=1.8, opacity=0.7)
       .encode(x=alt.X("x:Q", scale=XS), y=alt.Y("y:Q", scale=YS)),
    alt.Chart(d).mark_circle(size=75, opacity=0.9, stroke="white", strokeWidth=0.6).encode(
        x=alt.X("pwd_pool:Q", scale=XS, title="Population-weighted density (residents per km², log scale)",
                axis=alt.Axis(values=[1000, 2000, 5000, 10000, 20000], format=",")),
        y=alt.Y("realloc:Q", scale=YS, title="Job reallocation rate", axis=alt.Axis(labelExpr="datum.label + '%'")),
        color=alt.Color("group:N", scale=alt.Scale(domain=["London", "Rest of UK"], range=[ECO["red"], ECO["blue-light"]]),
                        legend=alt.Legend(title=None, orient="bottom")),
        tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("pwd_pool:Q", format=",.0f", title="Pop.-weighted density"),
                 alt.Tooltip("share_rural:Q", format=".0f", title="Rural share (%)"), alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
    *[alt.Chart(labs.iloc[[i]].rename(columns={"pop_weighted_density": "pwd_pool"})).mark_text(align=r["align"], dx=r["dx"], fontSize=9, color=TXT)
        .encode(x=alt.X("pwd_pool:Q", scale=XS), y=alt.Y("realloc:Q", scale=YS), text="short:N") for i, r in labs.iterrows()],
    alt.Chart(extra).mark_text(align="left", dx=8, fontSize=9, color=TXT).encode(x=alt.X("pwd_pool:Q", scale=XS), y=alt.Y("realloc:Q", scale=YS), text="short:N"),
).properties(title=alt.Title("Denser regions have more job churn, in and outside London",
    subtitle=[f"ITL2 regions, 2018–24: job reallocation rate and population-weighted density, both pooled over the period. Correlation {r_all:.2f}; {r_ex:.2f} excluding London.",
              "Population-weighted density: density of the small area the average resident lives in. Grey line = fit across all regions.", SOURCE]))
styles.save(ch, str(FIG), "reallocation_density_2018_2024", width=W, height=H)
ch.properties(width=W, height=H)

2018–24, 41 regions: r = 0.78, excl. London 0.50; UK 28.0%
Extra label: West Central Scotland 5363 26.6


alt.LayerChart(...)

In [5]:
# Rural share vs reallocation, pooled 2018-24, all 41 regions, coloured by nation (definitions of "rural" differ by nation)
dr = d.copy()   # 2018-24 frame from the density scatter
dr["area"] = np.where(LONDON(dr), "London", dr.itl2_code.str[:3].map({"TLL": "Wales", "TLM": "Scotland", "TLN": "Northern Ireland"}).fillna("Rest of England"))
AREAS = ["London", "Rest of England", "Wales", "Scotland", "Northern Ireland"]
ACOLS = [ECO["red"], ECO["blue-light"], ECO["turquoise"], ECO["blue-dark"], ECO["yellow"]]
r_r = np.corrcoef(dr.realloc, dr.share_rural)[0, 1]
ew = dr[dr.area.isin(["London", "Rest of England", "Wales"])]; r_ew = np.corrcoef(ew.realloc, ew.share_rural)[0, 1]
print(f"Rural share vs reallocation 2018–24: r = {r_r:.2f} (41 regions); England & Wales only {r_ew:.2f}")

W, H = 560, 400
XR = alt.Scale(domain=[-2, 60], nice=False)
YR = alt.Scale(domain=[float(np.floor(dr.realloc.min() - 1)), float(np.ceil(dr.realloc.max() + 1))], zero=False, nice=False)
b = np.polyfit(dr.share_rural, dr.realloc, 1); fx = np.array([0, dr.share_rural.max()])
labs = place_labels(dr.assign(group=dr.area), "share_rural", "realloc", XR, YR, W, H, max_labels=9, max_len=22, pad=2)
rur = alt.layer(
    alt.Chart(pd.DataFrame({"v": [uk]})).mark_rule(strokeDash=[2, 3], color=TXT, opacity=0.5).encode(y=alt.Y("v:Q", scale=YR)),
    alt.Chart(pd.DataFrame({"x": [XR.domain[1]], "y": [uk]})).mark_text(align="right", baseline="bottom", dx=-3, dy=-3, fontSize=10, color=TXT)
       .encode(x=alt.X("x:Q", scale=XR), y=alt.Y("y:Q", scale=YR), text=alt.value(f"UK {uk:.1f}%")),
    alt.Chart(pd.DataFrame({"x": fx, "y": np.polyval(b, fx)})).mark_line(color=GREY, strokeWidth=1.8, opacity=0.7)
       .encode(x=alt.X("x:Q", scale=XR), y=alt.Y("y:Q", scale=YR)),
    alt.Chart(dr).mark_circle(size=75, opacity=0.9, stroke="white", strokeWidth=0.6).encode(
        x=alt.X("share_rural:Q", scale=XR, title="Share of residents living in rural areas", axis=alt.Axis(values=list(range(0, 61, 10)), labelExpr="datum.label + '%'")),
        y=alt.Y("realloc:Q", scale=YR, title="Job reallocation rate", axis=alt.Axis(labelExpr="datum.label + '%'")),
        color=alt.Color("area:N", scale=alt.Scale(domain=AREAS, range=ACOLS), legend=alt.Legend(title=None, orient="bottom")),
        tooltip=[alt.Tooltip("itl2_name:N", title="Region"), alt.Tooltip("share_rural:Q", format=".0f", title="Rural share (%)"),
                 alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
    *[alt.Chart(labs.iloc[[i]]).mark_text(align=r["align"], dx=r["dx"], fontSize=9, color=TXT)
        .encode(x=alt.X("share_rural:Q", scale=XR), y=alt.Y("realloc:Q", scale=YR), text="short:N") for i, r in labs.iterrows()],
).properties(title=alt.Title("More rural regions have less job churn",
    subtitle=[f"ITL2 regions: share of residents in rural areas (Census) against job reallocation rate pooled 2018–24. Correlation {r_r:.2f}; {r_ew:.2f} for England and Wales only.",
              "Rural is defined differently in each nation: settlements under 10,000 people in England and Wales (2021), under 3,000 in Scotland (2022), under 5,000 in NI (2011),",
              "so Scottish shares are understated relative to the rest of the UK. Grey line = fit across all regions. London has no rural areas.", SOURCE]))
styles.save(rur, str(FIG), "reallocation_rural_share", width=W, height=H)
rur.properties(width=W, height=H)

Rural share vs reallocation 2018–24: r = -0.66 (41 regions); England & Wales only -0.68


alt.LayerChart(...)

## 3. Has the urban–rural gap changed over time?

Regions grouped by population-weighted density: **London** (5 regions) and the other regions in **quarters** by density (all 36 non-London regions, 9 each). Each group's rate is computed from its **summed counts** each year. Northern Ireland belongs to a quarter but is left out of the **lines**, so that each line's membership is the same every year (its series starts in 2011). It is included in the stacked bars, which cover 2011–24 only.

In [6]:
o = df[~LONDON(df)].copy()                     # quarters over all 36 non-London regions, incl. NI (9 each)
o["dgroup"] = pd.qcut(o.pop_weighted_density, 4, labels=["Least dense quarter", "Lower-middle quarter", "Upper-middle quarter", "Densest quarter"]).astype(str)
GROUP = {**dict(zip(o.itl2_code, o.dgroup)), **{c: "London" for c in df[LONDON(df)].itl2_code}}
ORDER = ["London", "Densest quarter", "Upper-middle quarter", "Lower-middle quarter", "Least dense quarter"]
COLS = [ECO["red"], ECO["blue-dark"], ECO["blue-light"], ECO["turquoise"], ECO["orange"]]
for g in ORDER[1:]:
    print(f"{g} ({(o.dgroup == g).sum()}): " + ", ".join(sorted(o[o.dgroup == g].itl2_name)))

ts = (jf[jf.itl2_code.isin(GROUP) & (jf.itl2_code != "TLN0")].assign(   # NI left out of the lines: its series starts in 2011
dgroup=lambda x: x.itl2_code.map(GROUP))
        .groupby(["dgroup", "year"])[["total_jc", "total_jd", "emp_avg"]].sum())
ts = (100 * (ts.total_jc + ts.total_jd) / ts.emp_avg).rename("realloc").reset_index()
piv = ts.pivot(index="year", columns="dgroup", values="realloc")[ORDER]
gap = (piv["Densest quarter"] - piv["Least dense quarter"])
print(f"\nGap densest - least dense quarter (pp): 2003–07 avg {gap.loc[2003:2007].mean():.1f}, 2011–15 {gap.loc[2011:2015].mean():.1f}, 2019–24 {gap.loc[2019:2024].mean():.1f}")
print(piv.loc[[2003, 2008, 2013, 2018, 2024]].round(1).to_string())

end = ts[ts.year == ts.year.max()]
lines = alt.layer(
    alt.Chart(ts).mark_line(strokeWidth=2.5).encode(
        x=alt.X("year:O", title=None, axis=alt.Axis(values=list(range(2004, 2025, 4)), labelAngle=0)),
        y=alt.Y("realloc:Q", scale=alt.Scale(zero=False), title="Reallocation rate (% of employment)"),
        color=alt.Color("dgroup:N", scale=alt.Scale(domain=ORDER, range=COLS), legend=None),
        tooltip=["dgroup:N", "year:O", alt.Tooltip("realloc:Q", format=".1f", title="Reallocation (%)")]),
    alt.Chart(end).mark_text(align="left", dx=6, fontSize=11).encode(
        x="year:O", y="realloc:Q", text="dgroup:N", color=alt.Color("dgroup:N", scale=alt.Scale(domain=ORDER, range=COLS), legend=None)),
).properties(title=alt.Title("Reallocation fell in dense and rural regions alike, so the gap persisted",
    subtitle=["Reallocation rate by density group: London, then the other ITL2 regions in quarters by population-weighted density (2021).",
              "Rates from summed counts each year. Northern Ireland excluded (series from 2011).", SOURCE]))
styles.save(lines, str(FIG), "reallocation_density_groups", width=560, height=360)
lines.properties(width=560, height=360)

Densest quarter (9): Bedfordshire and Hertfordshire, Eastern Scotland, Greater Manchester, Hampshire and Isle of Wight, Merseyside, Northumberland, and Tyne and Wear, West Central Scotland, West Midlands, West Yorkshire
Upper-middle quarter (9): Berkshire, Buckinghamshire and Oxfordshire, Derbyshire and Nottinghamshire, Essex, Gloucestershire, Wiltshire and Bath/Bristol area, Kent, Lancashire, Leicestershire, Rutland and Northamptonshire, South Yorkshire, Surrey, East and West Sussex
Lower-middle quarter (9): Cheshire, Devon, Dorset and Somerset, East Wales, East Yorkshire and Northern Lincolnshire, North Eastern Scotland, Northern Ireland, Southern Scotland, Tees Valley and Durham
Least dense quarter (9): Cornwall and Isles of Scilly, Cumbria, East Anglia, Herefordshire, Worcestershire and Warwickshire, Highlands and Islands, Lincolnshire, North Yorkshire, Shropshire and Staffordshire, West Wales and The Valleys

Gap densest - least dense quarter (pp): 2003–07 avg 1.4, 2011–15 3.3, 20

alt.LayerChart(...)

## 4. Which flows make the difference?

Pooled 2011–24 rates by density group, split into three pairs of flows: **entry and exit** of firms; incumbents **opening and closing sites**; incumbents' existing sites **expanding and contracting**. Each pair is (job creation + job destruction) as % of average employment, so the three sum to the reallocation rate.

In [7]:
comp = (jf[jf.year.between(*MAIN) & jf.itl2_code.isin(GROUP)].assign(dgroup=lambda x: x.itl2_code.map(GROUP))
          .groupby("dgroup")[FLOWS + ["emp_avg"]].sum())
cb = pd.DataFrame({k: 100 * comp[c].sum(axis=1) / comp.emp_avg for k, c in COMPONENTS.items()}).loc[ORDER]
cb["Total"] = cb.sum(axis=1)
print(cb.round(2).to_string())
gap_c = cb.loc["Densest quarter"] - cb.loc["Least dense quarter"]
print("\nDifference, densest minus least dense quarter (pp):", gap_c.round(2).to_dict())
print("  ... as % of the least dense quarter's rate:", (100 * gap_c / cb.loc["Least dense quarter"]).round(0).to_dict())

long = cb.drop(columns="Total").reset_index(names="dgroup").melt(id_vars="dgroup", var_name="flow", value_name="rate")
FORDER = list(COMPONENTS); FCOLS = [ECO["blue-dark"], ECO["turquoise"], ECO["blue-light"]]
long["order"] = long.flow.map({f: i for i, f in enumerate(FORDER)})
bars = alt.layer(
    alt.Chart(long).mark_bar(height=24).encode(
        y=alt.Y("dgroup:N", sort=ORDER, title=None),
        x=alt.X("rate:Q", stack="zero", title="Reallocation rate, 2011–24 (% of employment)"),
        color=alt.Color("flow:N", scale=alt.Scale(domain=FORDER, range=FCOLS), legend=alt.Legend(title=None, orient="bottom")),
        order=alt.Order("order:Q"),
        tooltip=["dgroup:N", "flow:N", alt.Tooltip("rate:Q", format=".1f", title="% of employment")]),
    alt.Chart(long).transform_stack(stack="rate", groupby=["dgroup"], as_=["x0", "x1"], sort=[alt.SortField("order")])
       .transform_calculate(mid="(datum.x0 + datum.x1) / 2").mark_text(color="white", fontSize=11).encode(
        y=alt.Y("dgroup:N", sort=ORDER), x="mid:Q", text=alt.Text("rate:Q", format=".1f")),
    alt.Chart(cb.reset_index(names="dgroup")).mark_text(align="left", dx=5, fontSize=11, color=TXT).encode(
        y=alt.Y("dgroup:N", sort=ORDER), x="Total:Q", text=alt.Text("Total:Q", format=".1f")),
).properties(title=alt.Title("Denser regions have more of every type of job flow",
    subtitle=["Reallocation by type of flow and density group, pooled 2011–24 (job creation + destruction, % of average employment).",
              f"Densest vs least dense quarter: +{gap_c['Entry and exit']:.1f} pp entry/exit, +{gap_c['Site openings and closures']:.1f} pp site openings/closures, "
              f"+{gap_c['Expansion and contraction']:.1f} pp expansion/contraction. Groups as in the previous chart; Northern Ireland included here.", SOURCE]))
styles.save(bars, str(FIG), "reallocation_density_components", width=560, height=270)
bars.properties(width=560, height=270)

                      Entry and exit  Site openings and closures  Expansion and contraction  Total
dgroup                                                                                            
London                          9.76                        8.46                      16.03  34.26
Densest quarter                 6.64                        7.34                      14.27  28.26
Upper-middle quarter            6.72                        7.10                      13.78  27.60
Lower-middle quarter            5.79                        6.38                      13.17  25.34
Least dense quarter             6.18                        6.41                      12.98  25.57

Difference, densest minus least dense quarter (pp): {'Entry and exit': 0.46, 'Site openings and closures': 0.93, 'Expansion and contraction': 1.29, 'Total': 2.68}
  ... as % of the least dense quarter's rate: {'Entry and exit': 7.0, 'Site openings and closures': 15.0, 'Expansion and contraction': 10.0, 'T

alt.LayerChart(...)

In [8]:
# 100% stacked: share of each group's reallocation from each type of flow, 2018-24.
# Groups: London, then the other 36 regions in quarters by population-weighted density pooled over 2018-24 (pwd_pool, as in the 2018-24 scatter).
SH_YRS = (2018, 2024)
nl_ = d[~LONDON(d)].copy()                      # d = 2018-24 frame from the density scatter (41 regions)
nl_["q"] = pd.qcut(nl_.pwd_pool, 4, labels=["Least dense quarter", "Lower-middle quarter", "Upper-middle quarter", "Densest quarter"]).astype(str)
GROUP_S = {**dict(zip(nl_.itl2_code, nl_.q)), **{c_: "London" for c_ in d[LONDON(d)].itl2_code}}
comp_s = (jf[jf.year.between(*SH_YRS) & jf.itl2_code.isin(GROUP_S)].assign(g=lambda x: x.itl2_code.map(GROUP_S))
            .groupby("g")[FLOWS + ["emp_avg"]].sum())
cb_s = pd.DataFrame({k: 100 * comp_s[c_].sum(axis=1) / comp_s.emp_avg for k, c_ in COMPONENTS.items()}).loc[ORDER]
sh = cb_s.div(cb_s.sum(axis=1), axis=0) * 100
print("Reallocation rate by group, 2018–24:", cb_s.sum(axis=1).round(1).to_dict()); print(sh.round(1).to_string())

def round100(row):                               # whole-number labels that add to exactly 100
    fl = np.floor(row.values); rem = row.values - fl; k = int(round(100 - fl.sum()))
    fl[np.argsort(-rem)[:k]] += 1; return pd.Series(fl, index=row.index)
lab_int = sh.apply(round100, axis=1)

ls = sh.reset_index(names="dgroup").melt(id_vars="dgroup", var_name="flow", value_name="share")
ls = ls.merge(lab_int.reset_index(names="dgroup").melt(id_vars="dgroup", var_name="flow", value_name="lab"), on=["dgroup", "flow"])
ls["order"] = ls.flow.map({f: i for i, f in enumerate(FORDER)})
share_bars = alt.layer(
    alt.Chart(ls).mark_bar(height=26).encode(
        y=alt.Y("dgroup:N", sort=ORDER, title=None, axis=alt.Axis(labelLimit=260)),
        x=alt.X("share:Q", stack="normalize", title=None, axis=alt.Axis(format="%", values=[0, 0.25, 0.5, 0.75, 1])),
        color=alt.Color("flow:N", scale=alt.Scale(domain=FORDER, range=FCOLS), legend=alt.Legend(title=None, orient="bottom")),
        order=alt.Order("order:Q"),
        tooltip=["dgroup:N", "flow:N", alt.Tooltip("share:Q", format=".1f", title="Share of reallocation (%)")]),
    alt.Chart(ls).transform_stack(stack="share", groupby=["dgroup"], as_=["x0", "x1"], sort=[alt.SortField("order")], offset="normalize")
       .transform_calculate(mid="(datum.x0 + datum.x1) / 2", txt="format(datum.lab, '.0f') + '%'")
       .mark_text(color="white", fontSize=11).encode(y=alt.Y("dgroup:N", sort=ORDER), x=alt.X("mid:Q"), text="txt:N"),
).properties(title=alt.Title("The mix of job flows is similar in dense and rural regions; London stands out on entry and exit",
    subtitle=["Share of the job reallocation rate from each type of flow, by density group, pooled 2018–24.",
              "Groups: London, then the other 36 ITL2 regions in quarters by population-weighted density (Northern Ireland included).", SOURCE]))
styles.save(share_bars, str(FIG), "reallocation_density_components_share", width=560, height=270)
share_bars.properties(width=560, height=270)

Reallocation rate by group, 2018–24: {'London': 34.0, 'Densest quarter': 27.7, 'Upper-middle quarter': 27.2, 'Lower-middle quarter': 25.3, 'Least dense quarter': 25.5}
                      Entry and exit  Site openings and closures  Expansion and contraction
g                                                                                          
London                          27.3                        25.3                       47.4
Densest quarter                 23.8                        26.2                       50.0
Upper-middle quarter            25.0                        26.1                       49.0
Lower-middle quarter            23.2                        25.7                       51.1
Least dense quarter             24.5                        25.9                       49.6


alt.LayerChart(...)

## 5. What this shows (and doesn't)

*(Numbers refer to the printed output above.)*

- **Density is linked to reallocation, and not just through London.** Unlike productivity, where the link largely disappeared outside London, the relationship with population-weighted density holds when London is dropped. It is stronger still for area density, holds for the rural share (including within England & Wales, where the rural definition is consistent), and survives controlling for productivity.
- **It is stable over time.** The correlation holds in each five-year window, and the urban–rural gap has been roughly constant while reallocation fell everywhere: the decline is not a rural phenomenon.
- **Mechanism:** the densest quarter has more of **every** type of flow than the least dense quarter (the middle quarters are not perfectly ordered for entry and exit). In points, most of the gap between the densest and least dense quarters is in incumbents' flows (expansion and contraction, and site openings and closures), because those flows are larger. Relative to their size, the gaps are similar, slightly larger for site openings and closures. Across regions, incumbent flows track density more closely than entry and exit (r ≈ 0.70 and 0.56 vs 0.28, excl. London). So thinner rural markets show up mainly as less expansion, contraction and branch opening by existing firms, not only as fewer start-ups. London stands out instead on **entry and exit** (9.8% vs 6–7%).
- **Caveats:**
  - **Industry mix:** rural economies have more agriculture, public sector and tourism, and fewer business services. An industry breakdown in the secure environment is needed to tell density effects apart from sector effects.
  - **Firm structure:** site openings and closures depend on multi-site firms, which are concentrated in cities.
  - **Rural definitions differ by nation:** Scotland's < 3k threshold makes its rural shares look lower. The density measures avoid this.
  - **Time coverage:** density is measured in one year (2021) and changes slowly, so this is cross-sectional evidence only.